# Conversor HULC → OpenSimula — Edificio PMT (ATECYR) — v2

**TFG — Conversión de modelos HULC a OpenSimula para simulación energética de edificios**

Versión de este cuaderno respecto al principal (`Demostracion_HULC_OpenSimula_edificio.ipynb`):

1. **Parche de un bug de OpenSimula** en el cálculo de factores de forma para la
   radiación de onda larga interior (`Space._create_ff_matrix`). Era la causa
   de que la buhardilla y los demás espacios no habitables salieran 10-12 °C
   más fríos que en HULC, lo que disparaba la demanda del edificio
   (calefacción ~2×, refrigeración ~0). Ver la explicación abajo. Se aplica
   importando `parche_opensimula_ff`.
2. **Tres correcciones del conversor** (ya en `hulc_geometria.py` /
   `hulc_to_opensimula.py`, no hace falta tocar nada en este cuaderno):
   - `incluir_unconditioned=False` por defecto → los `TYPE=UNCONDITIONED` de
     HULC (aquí P02_E05) no llevan sistema. En `P02_E05.txt` HULC le da 0 h de
     equipo y demanda 0.
   - Cuando el `FRAME-CONDUCT` que HULC guarda en un objeto `WINDOW` es MAYOR
     que el del marco de catálogo (porque HULC le ha sumado el puente del
     intercalario y el `porcentajeIncrementoU`), se usa ese valor efectivo con
     `R = 1/FRAME-CONDUCT` **sin restar películas** (no es aire-aire: 9,224 >
     1/0,17 = 5,88, el techo físico de una U aire-aire). El `FRAME-CONDUCT` de
     catálogo (`NAME-FRAME`) **sí** es aire-aire (EN ISO 10077) y se le siguen
     restando Rsi+Rse (§7.3 de la memoria). Las puertas ciegas copian el valor
     del catálogo → no se tocan.
   - `INF-METHOD = CRACK` (a secas): el `AIR-CHANGES/HR` del archivo es una
     referencia a presión de ensayo, no se aplica literal; el conversor deriva
     infiltración + ventilación de `BUILD-PARAMETERS.INF-FLOW`,
     `<valorImpulsionAire>` y el recuperador de `<datosVentilacion>`. *Este
     caso ATECYR es `CRACK-AIR-CHANGE`, no `CRACK`, así que aquí no cambia
     nada (se usa el `AIR-CHANGES/HR = 1` del archivo tal cual); sí cambia en
     las viviendas de los otros cuadernos.*
3. **La demanda del edificio se refiere a 65,97 m²** (P02_E01 + E02 + E03 + E04),
   la superficie sobre la que divide HULC. Recalculando la demanda a partir de
   las `Carga Total` de los `.txt` de HULC sobre esos 65,97 m² sale exactamente
   62,84 / 23,48 kWh/m²·año — la referencia.
4. **Comparación zona a zona** contra la salida horaria de HULC (`P0X_E0X.txt`).

Ninguno de estos cambios toca un valor de HULC ni falsea la geometría (la
cubierta sigue siendo la real inclinada, `aplanar_cubierta=False`).

## 1. Importar el conversor y aplicar el parche de OpenSimula

El conversor son dos módulos (`hulc_geometria.py`, `hulc_to_opensimula.py`).
`parche_opensimula_ff` se importa **antes** de crear la `Simulation`: aplica el
monkeypatch a `Space._create_ff_matrix` al importarse.

In [37]:
import opensimula as osm
import numpy as np
import json

# Recargar los módulos del conversor si se han editado:
import importlib, hulc_geometria, hulc_to_opensimula
importlib.reload(hulc_geometria)
importlib.reload(hulc_to_opensimula)
from hulc_to_opensimula import ctehexml_a_diccionario

# PARCHE del bug de factores de forma de OpenSimula (se aplica al importar).
import parche_opensimula_ff

print('Conversor y parche importados correctamente.')

Conversor y parche importados correctamente.


### El bug de OpenSimula que corrige el parche

`Space._create_ff_matrix` cierra la matriz de factores de forma con una
iteración que **no converge** cuando un plano del espacio (suelo o techo) está
partido en varias superficies. En la buhardilla `P03_E01` el "suelo" son 6
forjados de techo (uno por oficina de la planta de abajo): esas 6 superficies
son coplanares entre sí, la iteración agota las 500 iteraciones sin cerrar sus
filas y las deja sumando **0,707** en vez de 1,0. No hay normalización final.

Filas que no cierran (edificio de prueba):

| espacio | filas afectadas | suma |
|---|---|---|
| P03_E01 (buhardilla) | los 6 techos | 0,707 |
| P02_E06 | 2 superficies | 0,76 |
| P02_E04 (oficina) | 3 superficies | 0,956 |
| P01_E01 (cámara sanitaria) | ninguna | 1,000 |

El modelo de radiación de onda larga interior (`Building._create_LW_matrices_`)
usa esa matriz directamente, así que **~30 % de la radiación que emiten esas
superficies desaparece** (sumidero no físico). En la buhardilla, la suma de
`q_lwt` de todas sus superficies daba −1207 W de media (debería ser 0). Ese
calor que sube de las oficinas se "radiaba a la nada" → la buhardilla se
quedaba a 5,7 °C en vez de los 17,7 °C de HULC → arrastraba a todo el edificio.

El parche, tras la iteración original, si alguna fila no suma 1 recalcula la
matriz de ese espacio con un cierre biproporcional (Sinkhorn) que impone a la
vez `Σ fila = 1` y reciprocidad `Aᵢ·Fᵢⱼ = Aⱼ·Fⱼᵢ`, respetando qué superficies
se ven. No cambia ningún dato: corrige que la suma de factores de forma debe
ser 1.

Es un bug de OpenSimula (la geometría que genera el conversor es correcta y
HULC la digiere sin problema); pendiente de reportar al autor.

## 2. Leer HULC y generar el diccionario de OpenSimula

In [38]:
proyecto, informe = ctehexml_a_diccionario(
    "edificio_pmt_atecyr_sis3.ctehexml",
    ruta_biblioteca="BDCatalogo.bdc",
    incluir_geometria=True,
    archivo_met="zonaD3.met", tipo_met="MET",
    incluir_hvac=True,
    usar_horarios_cte=True,
    muros_verticales_underground=True,
    # Cubierta REAL inclinada, sin aplanar (aplanar_cubierta=True falsea la
    # geometría: sustituye los faldones por un techo plano).
    aplanar_cubierta=False,
    # incluir_unconditioned=False y usar_frame_conduct_hueco=True ya son el
    # valor por defecto del conversor.
)

proyecto["shadow_calculation"] = "INTERPOLATION"
print('Diccionario generado:', len(proyecto['components']), 'componentes.')

Diccionario generado: 215 componentes.


## 3. Comprobar las correcciones del conversor en el diccionario

- **P02_E05** no debe tener `HVAC_perfect_system` (es `UNCONDITIONED`).
- **`Opening_type` "VENTANA"** → marco `VENTANA__MARCO_HULC`, R = 1/9,224 =
  0,1084 (valor efectivo de HULC, **sin** resta de películas).
- **`Opening_type` "PUERTA"** → sigue con su marco de catálogo (madera), R con
  la resta de Rsi+Rse del §7.3: la puerta ciega copia el `FRAME-CONDUCT` del
  catálogo, no lleva intercalario.
- **Infiltración GESTORIA** = `1.0*i` (este caso es `INF-METHOD =
  CRACK-AIR-CHANGE`, no `CRACK`).

In [39]:
frames = {c['name']: c['thermal_resistance'] for c in proyecto['components'] if c['type'] == 'Frame'}
for c in proyecto['components']:
    if c['type'] == 'Opening_type':
        R = frames.get(c['frame'])
        conv = 'sin resta (efectivo HULC)' if 'MARCO_HULC' in c['frame'] else 'con resta Rsi+Rse (catalogo)'
        print(f"  Opening_type {c['name']:10s} -> {c['frame']:30s}  R={R:.4f}  1/R={1/R:5.2f}  [{conv}]")
    if c['type'] == 'Space_type' and 'GESTORIA_1' in c['name']:
        print(f"  {c['name']} infiltration = {c.get('infiltration')}")
print("  HVAC en:", [c['spaces'] for c in proyecto['components'] if c['type'] == 'HVAC_perfect_system'])

  Opening_type PUERTA     -> VER_Madera de densidad media alta  R=0.2845  1/R= 3.51  [con resta Rsi+Rse (catalogo)]
  Opening_type VENTANA    -> VENTANA__MARCO_HULC             R=0.1084  1/R= 9.23  [sin resta (efectivo HULC)]
  Uso_GESTORIA_1 infiltration = 1.0*i
  HVAC en: ['P02_E01', 'P02_E02', 'P02_E03', 'P02_E04']


## 4. Inspeccionar el diccionario generado

In [40]:
from collections import Counter
tipos = Counter(c.get('type') for c in proyecto['components'])
for tipo, n in sorted(tipos.items()):
    print(f"  {tipo:20s} {n}")

with open('proyecto_opensimula_generado_v2.json', 'w', encoding='utf-8') as f:
    json.dump(proyecto, f, indent=2, ensure_ascii=False)
print("\nGuardado en proyecto_opensimula_generado_v2.json")

  Building             1
  Building_surface     64
  Construction         20
  Day_schedule         26
  File_met             1
  Frame                6
  Glazing              2
  HVAC_perfect_system  4
  Material             30
  Opening              10
  Opening_type         2
  Space                8
  Space_type           6
  Week_schedule        20
  Year_schedule        15

Guardado en proyecto_opensimula_generado_v2.json


## 5. Crear el proyecto, comprobar y verificar el parche

`parche_opensimula_ff.verificar(pro)` imprime, por espacio, el min/max de la
suma de filas de `ff_matrix`. Con el parche activo **todas deben salir 1,000**
(sin el parche, P03_E01 y P02_E06 salen por debajo).

In [41]:
sim = osm.Simulation()
pro = sim.new_project('Edificio_PMT')
pro.read_dict(proyecto)
pro.check()

try:
    pro.show_3D(True)
except Exception as e:
    print(f"show_3D() ha fallado ({e!r}); seguimos sin la vista 3D.")

Reading project data from dictonary
Reading completed.
Checking project: Nombre del Proyecto
Checking completed.
Checking project: Nombre del Proyecto
Checking completed.


## 6. Simular el año completo

(~3 min. Tras simular, `ff_matrix` ya está construida y se puede verificar.)

In [42]:
pro.simulate()
print()
parche_opensimula_ff.verificar(pro)   # todas las filas deben salir 1,000

Calculating solar direct shadows ...
Simulating Nombre del Proyecto: ...


100%|██████████| 8760/8760 [02:17<00:00, 63.71step/s, n_iter=3]



espacio        n_sup  suma_filas min      max
P01_E01           14           1.000    1.000
P02_E01            8           1.000    1.000
P02_E02            7           1.000    1.000
P02_E03            6           1.000    1.000
P02_E04           10           1.000    1.000
P02_E05           20           1.000    1.000
P02_E06            7           1.000    1.000
P03_E01           23           1.000    1.000


## 7. Referencia de HULC: demanda y temperatura por zona

Se lee de los ficheros `P0X_E0X.txt` (salida horaria por zona de HULC, 8760
filas). Columna 12 = `Carga Total` en W/m² del espacio, con el signo de HULC:
**negativo = calefacción, positivo = refrigeración**. Columna 13 = temperatura
del aire.

- Demanda de calefacción de la zona = Σ (−carga) sobre las horas con carga < 0.
- Demanda de refrigeración = Σ carga sobre las horas con carga > 0.
- Demanda del edificio = Σ (demanda_zona × área_zona) / 65,97 m²
  (P02_E01 + E02 + E03 + E04). Da 62,84 / 23,48 → la referencia.

In [43]:
H = 8760
mes = np.zeros(H, dtype=int)
_k = 0
for _m, _d in enumerate([31,28,31,30,31,30,31,31,30,31,30,31]):
    mes[_k:_k+_d*24] = _m + 1
    _k += _d * 24

ZONAS = {"P02_E01": 16.2037, "P02_E02": 7.1726, "P02_E03": 10.1525, "P02_E04": 32.4463}
A_UTIL = sum(ZONAS.values())   # 65.97 m2, base de HULC

hulc = {}
for f in list(ZONAS) + ["P01_E01", "P02_E06", "P03_E01"]:
    A = np.array([[float(x) for x in ln.split(",")]
                  for ln in open(f + ".txt", encoding="latin-1").read().splitlines()[3:3+H]])
    ct, T = A[:, 12], A[:, 13]
    hulc[f] = {"cal": -np.where(ct < 0, ct, 0).sum() / 1000,
               "ref":  np.where(ct > 0, ct, 0).sum() / 1000,
               "T": T}

HULC_CALEF  = sum(hulc[z]["cal"] * a for z, a in ZONAS.items()) / A_UTIL
HULC_REFRIG = sum(hulc[z]["ref"] * a for z, a in ZONAS.items()) / A_UTIL
print(f"Referencia HULC (recalculada de los .txt, /{A_UTIL:.2f} m2):"
      f"  calef = {HULC_CALEF:.2f}   refrig = {HULC_REFRIG:.2f}  kWh/m2·año")

Referencia HULC (recalculada de los .txt, /65.98 m2):  calef = 62.85   refrig = 23.48  kWh/m2·año


## 8. Resultados: demanda zona a zona, OpenSimula vs HULC

`system_sensible_heat` de cada `Space` (viene del `HVAC_perfect_system`):
positivo = calefacción, negativo = refrigeración. Se separan sumando por signo,
paso de 1 h → Wh → /1000 → kWh.

In [44]:
def v(nombre, var):
    return np.array(pro.component(nombre).variable(var).values, dtype=float)

print(f"{'zona':10s} {'OS calef':>9s} {'HULC':>7s} {'OS refr':>9s} {'HULC':>7s}"
      f" {'OS Tjul':>8s} {'HULC':>7s}")
print("-" * 62)
tot_cal = tot_ref = 0.0
for z, a in ZONAS.items():
    s = v(z, "system_sensible_heat")
    T = v(z, "temperature")
    cal = np.where(s > 0, s, 0).sum() / 1000 / a
    ref = -np.where(s < 0, s, 0).sum() / 1000 / a
    tot_cal += cal * a
    tot_ref += ref * a
    print(f"{z:10s} {cal:9.1f} {hulc[z]['cal']:7.1f} {ref:9.1f} {hulc[z]['ref']:7.1f}"
          f" {T[mes==7].mean():8.1f} {hulc[z]['T'][mes==7].mean():7.1f}")

calef_edif  = tot_cal / A_UTIL
refrig_edif = tot_ref / A_UTIL
print("-" * 62)
print(f"{'EDIFICIO':10s} {calef_edif:9.1f} {HULC_CALEF:7.1f} {refrig_edif:9.1f} {HULC_REFRIG:7.1f}"
      f"      (/{A_UTIL:.2f} m2)")
print()
print(f"Ratio calefacción  OpenSimula / HULC: {calef_edif / HULC_CALEF:.2f}x")
print(f"Ratio refrigeración OpenSimula / HULC: {refrig_edif / HULC_REFRIG:.2f}x")

# P02_E05 (UNCONDITIONED en HULC): flota, sin demanda.
T5 = v("P02_E05", "temperature")
print(f"\nP02_E05 (sin sistema, flota):  T media {T5.mean():.1f} °C   "
      f"julio {T5[mes==7].mean():.1f} °C")

zona        OS calef    HULC   OS refr    HULC  OS Tjul    HULC
--------------------------------------------------------------
P02_E01         72.3    77.8      29.5    20.8     26.1    26.4
P02_E02         85.5    60.6      33.7    28.9     26.2    26.7
P02_E03         91.0    61.2      21.6    14.5     26.0    26.4
P02_E04         65.7    56.4      27.1    26.4     26.5    27.0
--------------------------------------------------------------
EDIFICIO        73.4    62.8      27.6    23.5      (/65.98 m2)

Ratio calefacción  OpenSimula / HULC: 1.17x
Ratio refrigeración OpenSimula / HULC: 1.17x

P02_E05 (sin sistema, flota):  T media 19.0 °C   julio 27.2 °C


## 9. Espacios no habitables — la verificación del parche

Con el bug, `P03_E01` salía a 5,7 °C de media (HULC: 17,7). Con el parche debe
quedar a ~1 °C de HULC. `P01_E01` ya estaba bien (sus factores de forma
cerraban) y apenas cambia.

In [45]:
print(f"{'espacio':10s} {'OS media':>9s} {'ene':>6s} {'jul':>6s}   "
      f"{'HULC media':>10s} {'ene':>6s} {'jul':>6s}")
print("-" * 62)
for b in ["P01_E01", "P02_E06", "P03_E01"]:
    T  = v(b, "temperature")
    Th = hulc[b]["T"]
    print(f"{b:10s} {T.mean():9.1f} {T[mes==1].mean():6.1f} {T[mes==7].mean():6.1f}   "
          f"{Th.mean():10.1f} {Th[mes==1].mean():6.1f} {Th[mes==7].mean():6.1f}")

espacio     OS media    ene    jul   HULC media    ene    jul
--------------------------------------------------------------
P01_E01         17.7   14.8   24.1         18.4   16.0   22.5
P02_E06         16.4    9.4   25.9         17.6    9.7   27.0
P03_E01         17.3   10.0   27.5         17.7    9.8   27.7


## 10. Estado

- **Parche de `ff_matrix`**: obligatorio. Corrige un fallo de conservación de
  energía de OpenSimula (la suma de factores de forma debe ser 1). Sin él nada
  de lo demás tiene sentido comparar. Pasa la demanda de 1,99× / 0,07× a
  **~1,17× / ~1,17×**, y las temperaturas de todos los espacios (habitables y no
  habitables) caen a ~1 °C de HULC.
- **Las 3 correcciones del conversor** (`incluir_unconditioned=False`, marco por
  `FRAME-CONDUCT` de cada `WINDOW`, `INF-METHOD=CRACK` no literal) ya están en
  `hulc_geometria.py` / `hulc_to_opensimula.py`. Aquí solo la primera y la
  segunda tienen efecto (este caso es `CRACK-AIR-CHANGE`, no `CRACK`).
- El objetivo no es clavar los números (son dos motores distintos) sino que las
  demandas se parezcan y que ningún espacio salga con temperatura no física.
- `INF-METHOD = CRACK-AIR-CHANGE`: HULC modula el 1 ren/h por viento/tiro
  (efectivo anual ~0,6). El conversor lo mantiene plano en 1,0. Es la única
  diferencia de método que queda; su efecto es de ~±0,2× en la calefacción.
- Pendiente: reportar el bug de `ff_matrix` al autor de OpenSimula.